In [2]:
import h5py
import pandas as pd
import glob
import numpy as np
import json

In [22]:
result_paths = ['/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/outputs/cont_GWTC3/prod_retrainedCE',\
    '/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/outputs/discrete_GWTC3/flow_retrainedCE',
    '/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/outputs/discrete_GWTC3/KDEs']

removing_keys = ['model_selection/detectable_samples']

for path in result_paths:
    results = glob.glob(path+'/*.hdf5')

    for result in results:
        print(result)
        with h5py.File(result,  "a") as f:
            try:
                del f[removing_keys[0]]
            except:
                continue
            print(result)
            print(f['model_selection'].keys())
            f.close()

/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/outputs/cont_GWTC3/prod_retrainedCE/output_seed27.hdf5
/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/outputs/cont_GWTC3/prod_retrainedCE/output_seed12.hdf5
/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/outputs/cont_GWTC3/prod_retrainedCE/output_seed94.hdf5
/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/outputs/cont_GWTC3/prod_retrainedCE/output_seed56.hdf5
/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/outputs/cont_GWTC3/prod_retrainedCE/output_seed39.hdf5
/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/outputs/discrete_GWTC3/flow_retrainedCE/output_seed12.hdf5
/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/outputs/discrete_GWTC3/flow_retrainedCE/output_seed27.hdf5
/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/outputs/discrete_GWTC3/flow_retrainedCE/output_seed94.hdf5
/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/outputs/discrete_GWTC3/flo

OSError: Unable to synchronously open file (file is already open for read-only)

In [32]:
#error with /data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/outputs/discrete_GWTC3/KDEs/output_seed39.hdf5
with h5py.File('/data/wiay/2297403c/amaze_model_select/output_seed39.hdf5',  "a") as f:
    del f[removing_keys[0]]
    print(result)
    print(f['model_selection'].keys())

/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/outputs/discrete_GWTC3/KDEs/output_seed39.hdf5
<KeysViewHDF5 ['lnprb', 'obsdata', 'raw_samples', 'samples']>


In [41]:
#writing flow mappings as JSON file
channels = ['CE', 'CHE', 'GC', 'NSC', 'SMT']

for channel in channels:
    mappings = np.load(f'/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/inputs/flow_models/mixed_models/{channel}_mappings.npy')
    print(mappings)
    channel_config = {'Mmax':mappings[0], 'Mup':mappings[1],'qmax':mappings[2],'qup':mappings[3], 'zmax':mappings[4], 'zup':mappings[5]}
    channel_json = {}
    channel_json[channel] = channel_config

    #write this channels config to file
    with open(f'/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/inputs/flow_models/mixed_models/flow_mappings.json', 'a') as f:
        json.dump(channel_json, f)

[ -0.50543649 100.          13.34709027   1.          14.03557119
  10.        ]
[ -0.48140259 100.          14.01969998   1.           8.6194462
  10.        ]
[  0.83147742 100.           6.90775528   1.001        4.22707981
  10.        ]
[  2.37249913 100.           9.04990962   1.           0.63633117
  10.        ]
[ -0.50663736 100.          11.73507779   1.           7.09703192
  10.        ]


In [53]:
#writing dataspace samps as hdf5 file
channels = ['CE', 'CHE', 'GC', 'NSC', 'SMT']
params = ['mchirp','q','chieff','z']

outdir='/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/plots/prod_091224/'
total_samps_ordered = np.load(f"{outdir}/data/flow_samps_dataspace.npy")
samps_filled = np.load(f'{outdir}/data/no_channelsamps_dataspace.npy')

with h5py.File(f'{outdir}/data/dataspace_samps.hdf5', 'w') as f:
    f.create_group("flow_samps")
    f.create_group("parametric_samps")

for cidx, channel in enumerate(channels):
    df = pd.DataFrame(total_samps_ordered[samps_filled[cidx-1]:samps_filled[cidx]], columns=params)
    df.to_hdf(f'{outdir}/data/dataspace_samps.hdf5', key=f'flow_samps/{channel}')

comb_intrins_samps = pd.read_hdf('/data/wiay/2297403c/GW_ChirpSim/binary_param_generation/60_intrins_samps_zmax1_35.hdf5', key='all_intrins_samps')
mchirp_samps = comb_intrins_samps['m1']*(comb_intrins_samps['q']**3/(1+comb_intrins_samps['q']))
chieff_samps = ((comb_intrins_samps['chi_1']*comb_intrins_samps['costilt_1'])+(comb_intrins_samps['q']*comb_intrins_samps['chi_2']*comb_intrins_samps['costilt_2']))\
/(1+comb_intrins_samps['q'])

PLPP_samps = [mchirp_samps, comb_intrins_samps['q'], chieff_samps, comb_intrins_samps['z']]
df = pd.DataFrame(np.array(PLPP_samps).T, columns=params)
df.to_hdf(f'{outdir}/data/dataspace_samps.hdf5', key=f'parametric_samps')
        

In [52]:
total_samps_ordered[samps_filled[cidx-1]:samps_filled[cidx]].shape

(62490, 4)

In [5]:
#writing corner flow/KDE samps as hdf5 file
channels = ['CE']
params = ['mchirp','q','chieff','z']

outdir='/data/wiay/2297403c/amaze_model_select/Nflows_AMAZE_paper/plots/prod_091224'
corner_samps_flow = np.load(f"{outdir}/data/CE_flows_cornersample.npy")
corner_samps_flow_test = np.load(f"{outdir}/data/CE_flows_cornersample_testCE.npy")
corner_samps_kde = np.load(f'{outdir}/data/CE_KDEs_cornersample.npy')

with h5py.File(f'{outdir}/data/emulation_samps.hdf5', 'w') as f:
    f.create_group("flow_samps")
    f.create_group("KDE_samps")

df = pd.DataFrame(corner_samps_flow, columns=params)
df.to_hdf(f'{outdir}/data/emulation_samps.hdf5', key=f'flow_samps')
df = pd.DataFrame(corner_samps_kde, columns=params)
df.to_hdf(f'{outdir}/data/emulation_samps.hdf5', key=f'KDE_samps')


with h5py.File(f'{outdir}/data/test_flow_samps.hdf5', 'w') as f:
    f.create_group("flow_samps")

df = pd.DataFrame(corner_samps_flow_test, columns=params)
df.to_hdf(f'{outdir}/data/emulation_samps.hdf5', key=f'flow_samps')
